In [ ]:
# Drive mount
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.8 MB/s eta 0:00:00


T4 GPUが利用可能かどうか確認

In [ ]:
# 1. GPU (T4等) が利用可能か自動チェック
import sys
import torch

if torch.cuda.is_available():
    device = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
    print(f"✅ GPUが有効です: {gpu_name}")
else:
    # CPUの場合は警告を出す
    print("⚠️ 警告: GPUが有効になっていません！")
    print("Colabの上部メニュー [ランタイム] -> [ランタイムのタイプを変更] から『T4 GPU』を選択してください。")

Colabで「Qwen2.5-7B」をColabで動かす  (70億のパラメータ)

In [ ]:
# 1. 必要なライブラリのインストール（bitsandbytesを追加）
!pip install -q transformers torch accelerate bitsandbytes
!pip install -q transformers accelerate bitsandbytes pandas
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# 4-bit量子化（メモリ削減）の設定
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# 7Bモデルを指定
model_name = "Qwen/Qwen2.5-7B-Instruct"

print("7Bモデルをダウンロード中...（約15分かかります）")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,  # 4-bitで軽量化して読み込み
    device_map="auto"
)

print("モデルの読み込み完了！")

7Bモデルをダウンロード中...（約15分かかります）


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

モデルの読み込み完了！


Qwenを最初にテストする

In [ ]:
# -------------------------
# 2. テストメッセージの作成
# -------------------------

# プロンプト（フォーマット指定）
messages = [
    {
        "role": "system",
        "content": "あなたは「Qwen2.5-7B」という優秀なAIアシスタントです。"
    },
    {
        "role": "user",
        "content": "日本語で、自己紹介を箇条書きにして、説明も加えてください。\n日本語で、得意な事を箇条書きにして、説明も加えてください。"
    }
]

# message ⇒ text に変換する
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
      outputs = model.generate(
        **inputs,
        max_new_tokens=300,      # 標準的な出力文字数
        do_sample=True,          # サイコロを振る（以下のtemperatureとtop_pを有効にする）
        temperature=0.7,         # 創造性と正確さのベストバランス
        top_p=0.95,              # 変な言葉をカットする（標準値）
        repetition_penalty=1.1,  # 同じ言葉の繰り返しを適度に防ぐ（必ず1.0より大きくする）
        pad_token_id=tokenizer.pad_token_id,
    )

result_text = tokenizer.decode(outputs[0], skip_special_tokens=True, clean_up_tokenization_spaces=False)

# AIの回答部分だけを抽出
response = result_text.split("### 応答:\n")[-1].strip()

print(response)
print("-----------------------\n")


system
あなたは「Qwen2.5-7B」という優秀なAIアシスタントです。
user
日本語で、自己紹介を箇条書きにして、説明も加えてください。
日本語で、得意な事を箇条書きにして、説明も加えてください。
assistant
### 自己紹介
私は「Qwen2.5-7B」です。大規模言語モデルとして設計され、多様な言語処理タスクに優れています。質問への回答から翻訳、文章生成まで幅広く対応できます。特に日本語の理解と生成には精通しています。

---

### 得意な事
1. **文脈理解**
   - 複雑な文脈を理解し、適切な応答を提供します。長文の質問や複数の文から情報を引き出す能力が高くなっています。

2. **多言語対応**
   - 日本語だけでなく、英語、中国語など多数の言語に対応しており、互いに翻訳や文法チェックを行うことができます。

3. **質疑応答**
   - さまざまな分野（科学、技術、医学など）に関する知識を持ち、詳しい回答を提供します。また、一般的な情報から特定のトピックについての深い洞察まで対応可能です。

4. **文章生成**
   - 短い文章から長いエッセイまで作成可能。ストーリー、ニュース記事、ブログ投稿など様々な形式での文章生成が得意です。

5. **翻訳**
   - 高品質な機械翻�
-----------------------



Driveをマウント

In [ ]:
# Drive mount
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import seaborn as sns
import pandas as pd
from matplotlib import pyplot as plt
#!pip install japanize-matplotlib
#import japanize_matplotlib

!pip install pdfplumber
!pip install pymupdf

pdfフアイルを読み込む

In [ ]:
import pdfplumber

# 1. pdfの読み込み
path = "/content/drive/MyDrive/38_自然言語処理/市民アンケート(2026)/2026_csv/"
file = "R6年度市民意識調査(P38).pdf"
pdf_file_path = path + file

# PDFファイルを開く
with pdfplumber.open(pdf_file_path) as pdf:
    # 1ページ目（プログラムの世界では0番目）を指定
    # 全ページ読み込む場合は for page in pdf.pages:
    target_page = pdf.pages[0]

    # ページ内のテキストを抽出（レイアウトをできるだけ維持してくれます）
    extracted_text = target_page.extract_text()

# 正しく読み込めているか画面に表示して確認
print("【PDFから抽出したテキスト】")
print(extracted_text)

PDFの特定のページを綺麗な画像ファイル（PNG）として保存するコード

In [ ]:
!pip install pymupdf

アンケート自由意見　"市民の声"　をQwenを使って分析する

In [ ]:
# ※「市民の声」

n = len(df)
opinions = df["市民の声"].dropna().tolist()

prompt_template = """以下の【市民の意見】について、指定の【出力フォーマット】に従って分析してください。

【出力フォーマット】
- 感情: （ポジティブ / ネガティブ / ニュートラル）
- 対象: （何についての意見かを具体的に）
- 特徴: （意見の背景にある市民のニーズを1文で）
- 解決：(市民のニーズにこたえる解決方法を提案してください)

【市民の意見】
{text_data}
"""

# ==========================================
# 4. ループ処理で1件ずつAIに分析させる
# ==========================================
results = []
print(f"\n--- 分析を開始します（合計 {len(opinions[:n])} 件） ---")

for i, opinion in enumerate(opinions[:n]):
    # プロンプトのひな形に意見を流し込む
    user_prompt = prompt_template.format(text_data=opinion)

    # Qwen-Instructモデル用のチャット形式に変換
    messages = [
        {"role": "system", "content": "あなたは優秀な自治体のデータアナリストです。指定されたフォーマットに従い、余計な挨拶は省いて分析結果のみを日本語で出力してください。"},
        {"role": "user", "content": user_prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # トークン化してGPUへ転送
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # AIによる推論（テキスト生成）
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=300,
        do_sample=False,
        repetition_penalty=1.0,
        pad_token_id=tokenizer.pad_token_id,
    )

    input_length = model_inputs.input_ids.shape[1]
    generated_tokens = generated_ids[0][input_length:]

    # 切り取った回答部分だけをテキストに変換（デコード）
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    print(f"\n▼ {i+1}件目: {opinion}")
    print(f"【AIの分析】\n{response.strip()}")

    results.append({
        "元の意見": opinion,
        "AIの分析": response.strip()
    })


--- 分析を開始します（合計 8 件） ---

▼ 1件目: 高坂駅から出るバスの本数が少ない。(特に土日、祝、夜23:00頃までの時間)デマンドタクシー等の公共交通サービスを充実させてほしい。
【AIの分析】
- 感情: ネガティブ
- 対象: 高坂駅のバス本数（特に土日、祝、夜23:00頃までの時間）
- 特徴: 市民の夜間・休日移動のニーズが満たされていない
- 解決: デマンドタクシーなどの代替手段を増やし、夜間・休日でも効率的な移動手段を提供する。

▼ 2件目: 小学生が夏休みに遊ぶことのできる施設が極端に少ないと感じます。暑い最中、外で遊ぶこともできず、ソーレやマーレにも入れないとなると、暑い日の遊び場が商業施設しかなくあまり毎日のようには連れて行けません。小学生でも使える安全で比較的利用料が安価な施設があると嬉しいです。
【AIの分析】
- 感情: ネガティブ
- 対象: 小学生が夏休みに遊べる施設の不足
- 特徴: 暑い日には安全で安価な施設が不足しているため、毎日連れて行けないという市民のニーズ
- 解決: 公共のプールや体育館を夏期限定で無料開放し、安全管理体制を強化する。また、地域の図書館や文化施設を開放して、学習や文化活動の場を提供する。

▼ 3件目: (地区センター)や公園に子供用の用具がありますが、これからは老人が多くなるので、何時でもふらっと使える用具があるといいと思います。(例えば、筋トレになる様な用具)
【AIの分析】
- 感情: ポジティブ
- 対象: 公園や地区センターの用具
- 特徴: 老人人口の増加に伴い、いつでも利用できる多様な用具の必要性
- 解決: 多目的な運動器具を設置し、時間制限なく利用できるようにする。また、地域住民の協力を得て、維持管理を行う。

▼ 4件目: スリーデーマーチだけに留めず、遊歩道を整備し(周辺町も巻き込んで)ウォーキングの聖地にする。歴史、文化施設、公園他を経由、歩道沿に休憩所、カフェ、やきとり屋、入浴施設、宿泊(キャンプ場)施設を設置。サイクリングロードの併設も有り。
【AIの分析】
- 感情: ポジティブ
- 対象: 游歩道の整備とウォーキングの聖地化
- 特徴: 市民は地域の歴史や文化を活用し、観光地としての価値を高めるニーズがある
- 解決: 地域全体の協力のもと、多様な施設

In [ ]:
# -----------------------
# テキストフアイルの保存
# ------------------------
path = "/content/drive/MyDrive/38_自然言語処理/市民アンケート(2026)/2026_csv/"
file = "R6東松山市【高坂丘陵地区】analysis_results.txt"

with open(path + file, "w", encoding="utf-8") as f:

    for i, opinion in enumerate(opinions):
        # ... (AIの実行や response を取得する処理) ...

        # ▼ 画面にも表示
        print(f"\n▼ {i+1}件目: {opinion}")
        print(f"【AIの分析】\n{response.strip()}")

        # ▼ 同じ内容をテキストファイルにも書き込む（\n は改行の意味）
        f.write(f"▼ {i+1}件目: {opinion}\n")
        f.write(f"【AIの分析】\n{response.strip()}\n\n")
        f.write("-" * 40 + "\n") # 見やすいように区切り線を引く

        # リストへの保存
        results.append({
            "元の意見": opinion,
            "AIの分析": response.strip()
        })


▼ 1件目: 高坂駅から出るバスの本数が少ない。(特に土日、祝、夜23:00頃までの時間)デマンドタクシー等の公共交通サービスを充実させてほしい。
【AIの分析】
- 感情: ニュートラル
- 対象: 市の開発や近代化政策
- 特徴: 市民は伝統的な価値や環境の保全を重視している。
- 解決: 近代化と伝統の調和を追求し、地域の歴史的文化資産を保護しながら進歩を推進する政策を実施する。

▼ 2件目: 小学生が夏休みに遊ぶことのできる施設が極端に少ないと感じます。暑い最中、外で遊ぶこともできず、ソーレやマーレにも入れないとなると、暑い日の遊び場が商業施設しかなくあまり毎日のようには連れて行けません。小学生でも使える安全で比較的利用料が安価な施設があると嬉しいです。
【AIの分析】
- 感情: ニュートラル
- 対象: 市の開発や近代化政策
- 特徴: 市民は伝統的な価値や環境の保全を重視している。
- 解決: 近代化と伝統の調和を追求し、地域の歴史的文化資産を保護しながら進歩を推進する政策を実施する。

▼ 3件目: (地区センター)や公園に子供用の用具がありますが、これからは老人が多くなるので、何時でもふらっと使える用具があるといいと思います。(例えば、筋トレになる様な用具)
【AIの分析】
- 感情: ニュートラル
- 対象: 市の開発や近代化政策
- 特徴: 市民は伝統的な価値や環境の保全を重視している。
- 解決: 近代化と伝統の調和を追求し、地域の歴史的文化資産を保護しながら進歩を推進する政策を実施する。

▼ 4件目: スリーデーマーチだけに留めず、遊歩道を整備し(周辺町も巻き込んで)ウォーキングの聖地にする。歴史、文化施設、公園他を経由、歩道沿に休憩所、カフェ、やきとり屋、入浴施設、宿泊(キャンプ場)施設を設置。サイクリングロードの併設も有り。
【AIの分析】
- 感情: ニュートラル
- 対象: 市の開発や近代化政策
- 特徴: 市民は伝統的な価値や環境の保全を重視している。
- 解決: 近代化と伝統の調和を追求し、地域の歴史的文化資産を保護しながら進歩を推進する政策を実施する。

▼ 5件目: 市からの補助などが分かるような発信が少ない。住民税が高い割に具体的な使い道、改善が分からない。
【AIの分析】
- 感情: ニュートラル
- 対象: 市の

In [ ]:
# ---------------------
# csvフアイルに保存する
# ---------------------
import csv
path = "/content/drive/MyDrive/38_自然言語処理/市民アンケート(2026)/2026_csv/"
name = "【高坂丘陵地区】Qwen-analysis_results.csv"
csv_file = path + name

with open(csv_file, mode="w", encoding="utf-8-sig", newline="") as f:

    # 辞書のキー（"元の意見", "AIの分析"）を列の見出し（ヘッダー）として設定
    fieldnames = ["元の意見", "AIの分析"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)

    # 1行目（ヘッダー）をファイルに書き込む
    writer.writeheader()

    writer.writerows(results)

print("CSVファイルへの保存が完了しました！")
